In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

import pandas as pd
import matplotlib.pyplot as plt
import unicodedata
import numpy as np
from scipy.stats import spearmanr
import openpyxl
from data_processing import *
from montecarlo import*
from sensitivity import*

In [ ]:
# ------------------------------------------------------------------------------
# 6. CORRIDA COMPLETA
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    FILE_ACTIVITY = "C:/Users/Pame/Desktop/carbon_uncertainty/data/data_activity.xlsx"
    FILE_UNCERTAINTY = "C:/Users/Pame/Desktop/carbon_uncertainty/data/uncertainty_dist_act_FE.xlsx"
    SEED=12345
    N_ITER = 200000
    
    # --- 1. matching ---
    df, sin_match, discrepancias = construir_dataframe(FILE_ACTIVITY, FILE_UNCERTAINTY)
    print(f"Items emparejados: {len(df)}")
    if sin_match:
        print("\nATENCION - actividades sin match (revisar nombres):")
        for r, fuente, actividad in sin_match:
            print(f"  fila {r}: '{actividad}' (fuente: {fuente})")
    if not discrepancias.empty:
        print("\nATENCION - FE de data_activity no coincide con uncertainty_dist_act_FE:")
        print(discrepancias[["actividad", "fe_reportado_en_actividad", "fe_central"]].to_string(index=False))
    n_fuzzy = (~df["match_exacto"]).sum()
    if n_fuzzy:
        print(f"\n{n_fuzzy} filas matchearon por prefijo (no exacto) -- auditar antes de confiar en fe_group:")
        print(df.loc[~df["match_exacto"], ["actividad", "fuente", "fe_group"]].to_string(index=False))

    print()
    df = afinar_fe_group(df)

    df["emision_central_kg"] = df["fe_central"].fillna(0) * df["cantidad"].fillna(0)
    df["emision_central_tn"] = df["emision_central_kg"] / 1000

    # --- 2. Monte Carlo: ingenuo vs. correlacionado ---
    rng_naive = np.random.default_rng(SEED)
    emis_naive = simular_montecarlo(df, N_ITER, rng_naive, correlacionar_fe=False)
    resumen_naive = resumir(df, emis_naive)

    rng_corr = np.random.default_rng(SEED)
    emis_corr = simular_montecarlo(df, N_ITER, rng_corr, correlacionar_fe=True)
    resumen_corr = resumir(df, emis_corr)

    print("\nMC ingenuo        (TOTAL):", resumen_naive["TOTAL"])
    print("MC correlacionado (TOTAL):", resumen_corr["TOTAL"])
    ancho_naive = resumen_naive["TOTAL"]["p97_5"] - resumen_naive["TOTAL"]["p2_5"]
    ancho_corr = resumen_corr["TOTAL"]["p97_5"] - resumen_corr["TOTAL"]["p2_5"]
    print(f"Ancho intervalo 95% -- ingenuo: {ancho_naive:.1f} tn | "
          f"correlacionado: {ancho_corr:.1f} tn ({ancho_corr/ancho_naive:.2f}x)")

    # --- 2b. Tier 1 analítico, para comparar contra los dos anteriores ---
    resumen_tier1 = propagacion_analitica(df)
    print("Tier 1 analítico  (TOTAL):", resumen_tier1)
    png_comparacion = graficar_comparacion_metodos(resumen_tier1, resumen_naive["TOTAL"], resumen_corr["TOTAL"])
    print(f"Gráfico comparativo guardado en: {png_comparacion}")

    # --- 3. sensibilidad (sobre la corrida correlacionada) ---
    tabla_sens = sensibilidad(df, emis_corr)
    tabla_sens_grupo = sensibilidad_por_grupo(df, emis_corr)

    print("\nTop 10 actividades por contribución a la INCERTIDUMBRE (no a la emisión media):")
    print(tabla_sens[["actividad", "fuente", "emision_media_pct_total",
                       "contribucion_incertidumbre_pct", "indice_riesgo"]]
          .head(10).to_string(index=False))

    print("\nTop 10 grupos de FE por contribución a la incertidumbre:")
    print(tabla_sens_grupo[["fe_group", "n_actividades", "contribucion_incertidumbre_pct"]]
          .head(10).to_string(index=False))

    png_path = graficar_sensibilidad(tabla_sens)
    print(f"\nGráfico guardado en: {png_path}")


In [ ]:
valor_fe = df["fe_central"].round(10).astype(str) + "_" + df["unidad_fe"].astype(str).str.strip()
grupos_por_valor = df.groupby(valor_fe)["fe_group"].nunique()
sospechosos = grupos_por_valor[grupos_por_valor > 1]

print(f"{len(sospechosos)} valores de FE que aparecen bajo MÁS DE UN fe_group distinto (correlación perdida):")
for valor, n in sospechosos.items():
    print(df.loc[valor_fe == valor, ["actividad", "fuente", "fe_group", "fe_central", "unidad_fe"]].to_string(index=False), "\n")

In [ ]:
print(df.groupby("fe_group").size().sort_values(ascending=False).head(10))

In [ ]:
tabla_grupo = sensibilidad_por_grupo(df, emis_corr)
print(tabla_grupo.head(10).to_string(index=False))